In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import papermill as pm
from datetime import datetime, timedelta
from tqdm import tqdm

## 1.1

In [24]:
from pathlib import Path

DB_PATH = Path("data/data.db")

print("Database exists:", DB_PATH.exists())

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)
conn.close()

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()

Database exists: True
Rows: 397711
Columns: ['ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker']


,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  str    
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  str    
dtypes: float64(5), int64(1), str(2)
memory usage: 24.3 MB


In [26]:
df.describe()

,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


### Column meanings

1. `ts`: Timestamp (observation date)
2. `ticker`: Stock's trading symbol
3. `open`: Opening stock price on trading day
4. `high`: Highest stock price on trading day
5. `low`: Lowest stock price on trading day
6. `close`: Closing stock price on trading day
7. `adj_closer`: Adjusted closing price (accoundting for adjustments like stock splits and dividends)
8. `volume`: Number of shared traded during the trading day

In [27]:
# Convert the date column to datetime for date analysis
df["ts"] = pd.to_datetime(df["ts"], errors="coerce")

# Display date range
print("Earliest trading date:", df["ts"].min())
print("Latest trading date:", df["ts"].max())

# Display ticker information
tickers = sorted(df["ticker"].dropna().unique())

print("\nNumber of unique tickers:", len(tickers))
print("Tickers:")
print(tickers)

Earliest trading date: 1962-01-02 00:00:00
Latest trading date: 2024-12-06 00:00:00

Number of unique tickers: 150
Tickers:
['<a target="_blank"><span style="color: blue">AAPL</span></a>', '<a target="_blank"><span style="color: blue">AMGN</span></a>', '<a target="_blank"><span style="color: blue">AMZN</span></a>', '<a target="_blank"><span style="color: blue">AXP</span></a>', '<a target="_blank"><span style="color: blue">BA</span></a>', '<a target="_blank"><span style="color: blue">CAT</span></a>', '<a target="_blank"><span style="color: blue">CRM</span></a>', '<a target="_blank"><span style="color: blue">CSCO</span></a>', '<a target="_blank"><span style="color: blue">CVX</span></a>', '<a target="_blank"><span style="color: blue">DIS</span></a>', '<a target="_blank"><span style="color: blue">GS</span></a>', '<a target="_blank"><span style="color: blue">HD</span></a>', '<a target="_blank"><span style="color: blue">HON</span></a>', '<a target="_blank"><span style="color: blue">IBM</span

## 1.2

In [28]:
diagnostic_df = df.copy()

diagnostic_df["ts"] = pd.to_datetime(diagnostic_df["ts"], errors="coerce")

price_cols = ["open", "high", "low", "close", "adj_close"]

print("Missing values in each column:")
print(diagnostic_df.isna().sum())

print("\nMissing-value percentages:")
print((diagnostic_df.isna().mean() * 100))

Missing values in each column:
ts               0
open             0
high             0
low              0
close            0
adj_close    19855
volume           0
ticker           0
dtype: int64

Missing-value percentages:
ts           0.000000
open         0.000000
high         0.000000
low          0.000000
close        0.000000
adj_close    4.992319
volume       0.000000
ticker       0.000000
dtype: float64


In [29]:
conn = sqlite3.connect(DB_PATH)

print("ohlc table columns:")
print(pd.read_sql_query("PRAGMA table_info(ohlc)", conn))

conn.close()

ohlc table columns:
   cid       name    type  notnull dflt_value  pk
0    0         ts    DATE        0       None   0
1    1       open   FLOAT        0       None   0
2    2       high   FLOAT        0       None   0
3    3        low   FLOAT        0       None   0
4    4      close   FLOAT        0       None   0
5    5  adj_close   FLOAT        0       None   0
6    6     volume  BIGINT        0       None   0
7    7     ticker    TEXT        0       None   0


In [33]:

def detect_missing_values(data):
    issues = []

    for col in data.columns:
        missing_rows = data[data[col].isna()]

        for idx, row in missing_rows.iterrows():
            issues.append({
                "ticker": row.get("ticker", None),
                "issue_type": "missing_value",
                "issue_date": row.get("ts", None),
                "description": f"Missing value in column '{col}' (row index {idx}).",
                "severity": "medium"
            })

    return issues


def detect_missing_trading_days(data, reference_data):
    target = data[["ticker", "ts"]].dropna().copy()
    reference = reference_data[["ticker", "ts"]].dropna().copy()

    target["ts"] = pd.to_datetime(target["ts"], errors="coerce")
    reference["ts"] = pd.to_datetime(reference["ts"], errors="coerce")

    target = target.dropna(subset=["ts"]).drop_duplicates(["ticker", "ts"])
    reference = reference.dropna(subset=["ts"]).drop_duplicates(["ticker", "ts"])

    issues = []

    for ticker, group in target.groupby("ticker"):
        start = group["ts"].min()
        end = group["ts"].max()

        observed_dates = set(group["ts"])

        expected_dates = set(
            reference.loc[
                (reference["ticker"] == ticker)
                & reference["ts"].between(start, end),
                "ts"
            ]
        )

        missing_dates = sorted(expected_dates - observed_dates)

        for missing_date in missing_dates:
            issues.append({
                "ticker": ticker,
                "issue_type": "missing_trading_day",
                "issue_date": missing_date,
                "description": (
                    f"Ticker {ticker} is missing a trading day on "
                    f"{missing_date.date()} between "
                    f"{start.date()} and {end.date()}."
                ),
                "severity": "medium"
            })

    return issues


def summarize_missing_days(data, reference_data):
    target = data[["ticker", "ts"]].dropna().copy()
    reference = reference_data[["ticker", "ts"]].dropna().copy()

    target["ts"] = pd.to_datetime(target["ts"], errors="coerce")
    reference["ts"] = pd.to_datetime(reference["ts"], errors="coerce")

    target = target.dropna(subset=["ts"]).drop_duplicates(["ticker", "ts"])
    reference = reference.dropna(subset=["ts"]).drop_duplicates(["ticker", "ts"])

    summaries = []

    for ticker, group in target.groupby("ticker"):
        start = group["ts"].min()
        end = group["ts"].max()

        observed_dates = set(group["ts"])

        expected_dates = set(
            reference.loc[
                (reference["ticker"] == ticker)
                & reference["ts"].between(start, end),
                "ts"
            ]
        )

        count = len(expected_dates - observed_dates)

        summaries.append({
            "ticker": ticker,
            "missing_trading_days": count,
            "summary": (
                f"Ticker {ticker} is missing {count} trading days "
                f"between {start.date()} and {end.date()}."
            )
        })

    return pd.DataFrame(
        summaries,
        columns=["ticker", "missing_trading_days", "summary"]
    )

In [34]:
import re
from html import unescape

conn = sqlite3.connect(DB_PATH)
reference_df = pd.read_sql_query("SELECT ticker, ts FROM ohlc", conn)
conn.close()

def normalize_ticker(value):
    if pd.isna(value):
        return value

    value = unescape(str(value))
    value = re.sub(r"<[^>]+>", "", value)

    return value.strip().upper()

diagnostic_df["ticker"] = diagnostic_df["ticker"].apply(normalize_ticker)
reference_df["ticker"] = reference_df["ticker"].apply(normalize_ticker)

In [38]:
reference_df["ts"] = pd.to_datetime(reference_df["ts"], errors="coerce")

missing_value_issues = detect_missing_values(diagnostic_df)
missing_day_issues = summarize_missing_days(diagnostic_df, reference_df)
missing_trading_days = detect_missing_trading_days(diagnostic_df, reference_df)

print("Missing-value issues:", len(missing_value_issues))
missing_day_issues.head()

Missing-value issues: 19855


AttributeError: 'list' object has no attribute 'head'

In [39]:
report_columns = [
    "ticker",
    "issue_type",
    "issue_date",
    "description",
    "severity"
]

def detect_return_outliers(data):
    issues = []

    work = data.copy()
    work["ts"] = pd.to_datetime(work["ts"], errors="coerce")
    work["close"] = pd.to_numeric(work["close"], errors="coerce")
    work = work.sort_values(["ticker", "ts"])
    work["prev_close"] = work.groupby("ticker")["close"].shift(1)
    work["daily_return"] = ((work["close"] - work["prev_close"]) / work["prev_close"])
    work.loc[
        ~np.isfinite(work["daily_return"]),
        "daily_return"
    ] = np.nan

    for ticker, group in work.groupby("ticker"):
        valid_returns = group["daily_return"].dropna()

        if len(valid_returns) < 2:
            continue

        mean_return = valid_returns.mean()
        std_return = valid_returns.std(ddof=1)

        if not np.isfinite(std_return) or std_return == 0:
            continue

        outliers = group[
            group["daily_return"].notna()
            & (
                (group["daily_return"] - mean_return).abs()
                > 3 * std_return
            )
        ]

        for _, row in outliers.iterrows():
            issues.append({
                "ticker": ticker,
                "issue_type": "return_outlier",
                "issue_date": row["ts"],
                "description": (
                    f"Daily return {row['daily_return']:.4%} is more "
                    f"than 3 sample standard deviations from the "
                    f"ticker's mean return ({mean_return:.4%}; "
                    f"sample SD {std_return:.4%}). Review; do not "
                    f"automatically delete."
                ),
                "severity": "medium"
            })

    return issues


return_outlier_issues = detect_return_outliers(diagnostic_df)

print("Number of return outliers:", len(return_outlier_issues))
display(pd.DataFrame(return_outlier_issues, columns=report_columns).head(10))

Number of return outliers: 5741


,ticker,issue_type,issue_date,description,severity
0,AAPL,return_outlier,1980-12-26,Daily return 9.2288% is more than 3 sample sta...,medium
1,AAPL,return_outlier,1981-04-02,Daily return 8.7659% is more than 3 sample sta...,medium
2,AAPL,return_outlier,1981-09-25,Daily return -12.9776% is more than 3 sample s...,medium
3,AAPL,return_outlier,1981-10-02,Daily return 8.1977% is more than 3 sample sta...,medium
4,AAPL,return_outlier,1981-12-17,Daily return 8.3340% is more than 3 sample sta...,medium
5,AAPL,return_outlier,1981-12-18,Daily return 8.2835% is more than 3 sample sta...,medium
6,AAPL,return_outlier,1982-03-19,Daily return 9.0173% is more than 3 sample sta...,medium
7,AAPL,return_outlier,1982-11-22,Daily return -8.9089% is more than 3 sample st...,medium
8,AAPL,return_outlier,1982-11-30,Daily return 10.3871% is more than 3 sample st...,medium
9,AAPL,return_outlier,1983-01-20,Daily return 11.1518% is more than 3 sample st...,medium


,ticker,issue_type,issue_date,description,severity


In [40]:
display(pd.DataFrame(missing_trading_days, columns=report_columns).head(10))

,ticker,issue_type,issue_date,description,severity


In [41]:

def detect_price_and_ohlc_issues(data):
    issues = []

    price_cols = ["open", "high", "low", "close", "adj_close"]

    work = data.copy()
    work["ts"] = pd.to_datetime(work["ts"], errors="coerce")

    for col in price_cols + ["volume"]:
        work[col] = pd.to_numeric(work[col], errors="coerce")

    for idx, row in work.iterrows():
        ticker = row.get("ticker")
        date = row.get("ts")
        for col in price_cols:
            value = row[col]

            if pd.notna(value) and (
                not np.isfinite(value) or value <= 0
            ):
                issues.append({
                    "ticker": ticker,
                    "issue_type": "invalid_price",
                    "issue_date": date,
                    "description": (
                        f"Column '{col}' has invalid price value "
                        f"{value}; prices must be positive and finite."
                    ),
                    "severity": "high"
                })

        high = row["high"]
        low = row["low"]
        close = row["close"]
        open_price = row["open"]

        if pd.notna(high) and pd.notna(close) and high < close:
            issues.append({
                "ticker": ticker,
                "issue_type": "impossible_ohlc",
                "issue_date": date,
                "description": f"High ({high}) is below close ({close}).",
                "severity": "high"
            })

        if pd.notna(low) and pd.notna(close) and low > close:
            issues.append({
                "ticker": ticker,
                "issue_type": "impossible_ohlc",
                "issue_date": date,
                "description": f"Low ({low}) is above close ({close}).",
                "severity": "high"
            })

        if all(pd.notna(row[c]) for c in ["low", "open", "high"]):
            if not (low <= open_price <= high):
                issues.append({
                    "ticker": ticker,
                    "issue_type": "ohlc_inconsistency",
                    "issue_date": date,
                    "description": (
                        f"Open ({open_price}) is outside "
                        f"the low-high range [{low}, {high}]."
                    ),
                    "severity": "high"
                })

        if all(pd.notna(row[c]) for c in ["low", "close", "high"]):
            if not (low <= close <= high):
                already_flagged = (
                    pd.notna(high) and pd.notna(close) and high < close
                ) or (
                    pd.notna(low) and pd.notna(close) and low > close
                )

                if not already_flagged:
                    issues.append({
                        "ticker": ticker,
                        "issue_type": "ohlc_inconsistency",
                        "issue_date": date,
                        "description": (
                            f"Close ({close}) is outside "
                            f"the low-high range [{low}, {high}]."
                        ),
                        "severity": "high"
                    })

        volume = row["volume"]

        if pd.notna(volume) and volume == 0:
            issues.append({
                "ticker": ticker,
                "issue_type": "zero_volume",
                "issue_date": date,
                "description": "Trading volume is zero; review the observation.",
                "severity": "low"
            })

        elif pd.notna(volume) and (
            not np.isfinite(volume) or volume < 0
        ):
            issues.append({
                "ticker": ticker,
                "issue_type": "invalid_volume",
                "issue_date": date,
                "description": f"Volume is invalid: {volume}.",
                "severity": "high"
            })

    return issues


price_ohlc_issues = detect_price_and_ohlc_issues(diagnostic_df)

print("Number of price/OHLC issues:", len(price_ohlc_issues))
display(pd.DataFrame(price_ohlc_issues, columns=report_columns).head(10))

Number of price/OHLC issues: 91


,ticker,issue_type,issue_date,description,severity
0,TRV,zero_volume,1978-12-20,Trading volume is zero; review the observation.,low
1,WMT,zero_volume,1975-05-01,Trading volume is zero; review the observation.,low
2,WMT,zero_volume,1973-09-10,Trading volume is zero; review the observation.,low
3,JNJ,zero_volume,1962-09-28,Trading volume is zero; review the observation.,low
4,WMT,zero_volume,1974-04-10,Trading volume is zero; review the observation.,low
5,WMT,zero_volume,1974-02-19,Trading volume is zero; review the observation.,low
6,WMT,zero_volume,1974-07-12,Trading volume is zero; review the observation.,low
7,WMT,zero_volume,1974-03-31,Trading volume is zero; review the observation.,low
8,WMT,zero_volume,1974-08-09,Trading volume is zero; review the observation.,low
9,JNJ,zero_volume,1967-06-14,Trading volume is zero; review the observation.,low


### Severity issues

High: Issues that make price data invalid or internally contradictory, such as nonpositive or non-finite prices, or impossible relationships (like high < low). These require investigation before the data is used.

Medium: Missing trading days and unusually large daily returns. These may affect analysis and should be reviewed, but they are not automatically errors.

Low: Zero trading volume, which may indicate stale or incomplete data but can also be legitimate in some circumstances.

In [43]:
all_issues = (
    missing_value_issues
    + missing_trading_days
    + return_outlier_issues
    + price_ohlc_issues
)

quality_report_before = pd.DataFrame(
    all_issues,
    columns=report_columns
)

Path("reports").mkdir(parents=True, exist_ok=True)

quality_report_before.to_csv(
    "reports/data_quality_report_before_repair.csv",
    index=False
)

print("Issues before repair:", len(quality_report_before))
display(quality_report_before.head(20))

Issues before repair: 25687


,ticker,issue_type,issue_date,description,severity
0,KO,missing_value,2024-04-16,Missing value in column 'adj_close' (row index...,medium
1,NVDA,missing_value,2018-06-12,Missing value in column 'adj_close' (row index...,medium
2,HD,missing_value,2013-04-16,Missing value in column 'adj_close' (row index...,medium
3,PG,missing_value,2015-03-20,Missing value in column 'adj_close' (row index...,medium
4,JPM,missing_value,2018-03-13,Missing value in column 'adj_close' (row index...,medium
5,NVDA,missing_value,2000-05-15,Missing value in column 'adj_close' (row index...,medium
6,CSCO,missing_value,2001-10-02,Missing value in column 'adj_close' (row index...,medium
7,MMM,missing_value,1975-07-03,Missing value in column 'adj_close' (row index...,medium
8,IBM,missing_value,2019-06-21,Missing value in column 'adj_close' (row index...,medium
9,CSCO,missing_value,2006-01-09,Missing value in column 'adj_close' (row index...,medium


## 1.3

In [44]:
import re
from html import unescape

In [47]:
raw_df = df.copy()

print("Original rows:", len(raw_df))
print("Original columns:", raw_df.columns.tolist())
print("Reference columns:", reference_df.columns.tolist())

Original rows: 397711
Original columns: ['ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker']
Reference columns: ['ticker', 'ts']


In [48]:
for frame in [raw_df, reference_df]:
    frame["ticker"] = frame["ticker"].apply(normalize_ticker)
    frame["ts"] = pd.to_datetime(
        frame["ts"], errors="coerce"
    ).dt.strftime("%Y-%m-%d")

all_price_cols = price_cols + ["adj_close"]
work_df = raw_df.copy()

for col in all_price_cols + ["volume"]:
    work_df[col] = pd.to_numeric(work_df[col], errors="coerce")

for col in price_cols:
    work_df[f"repaired_{col}"] = False

work_df["repaired_adj_close"] = False
work_df["volume_filled_zero"] = False
work_df["extreme_return_flag"] = False
work_df["repair_notes"] = ""

print("Working rows:", len(work_df))

Working rows: 397711


In [49]:
all_raw_missing = work_df[price_cols].isna().all(axis=1)

dropped_all_raw_missing = work_df.loc[all_raw_missing].copy()
work_df = work_df.loc[~all_raw_missing].copy()

print("Rows dropped because all raw prices were missing:",
      len(dropped_all_raw_missing))

work_df = work_df.sort_values(
    ["ticker", "ts"]
).reset_index(drop=True)

for col in price_cols:
    work_df[f"repaired_{col}"] = work_df[col].isna()

work_df[price_cols] = (
    work_df.groupby("ticker", sort=False)[price_cols].ffill()
)

for col in price_cols:
    mask = work_df[f"repaired_{col}"]
    work_df.loc[mask, "repair_notes"] += (
        f"Forward-filled {col}; "
    )

work_df["volume_filled_zero"] = work_df["volume"].isna()
work_df.loc[work_df["volume_filled_zero"], "repair_notes"] += (
    "Unknown volume filled with 0; original volume was missing; "
)
work_df["volume"] = work_df["volume"].fillna(0)

print("Missing raw prices after forward-fill:")
display(work_df[price_cols].isna().sum())

print("Missing volume after fill:", work_df["volume"].isna().sum())

Rows dropped because all raw prices were missing: 0
Missing raw prices after forward-fill:


open         0
high         0
low          0
close        0
adj_close    1
dtype: int64

Missing volume after fill: 0


In [50]:
return_outlier_issues = detect_return_outliers(work_df)

outlier_keys = {
    (issue["ticker"], pd.Timestamp(issue["issue_date"]).strftime("%Y-%m-%d"))
    for issue in return_outlier_issues
    if pd.notna(issue["issue_date"])
}

work_df["extreme_return_flag"] = [
    (ticker, date) in outlier_keys
    for ticker, date in zip(work_df["ticker"], work_df["ts"])
]

In [52]:
price_ohlc_issues = detect_price_and_ohlc_issues(work_df)

invalid_keys = {
    (issue["ticker"], pd.Timestamp(issue["issue_date"]).strftime("%Y-%m-%d"))
    for issue in price_ohlc_issues
    if issue["issue_type"] in [
        "invalid_price",
        "impossible_ohlc",
        "ohlc_inconsistency"
    ]
    and pd.notna(issue["issue_date"])
}

invalid_rows = [
    (ticker, date) in invalid_keys
    for ticker, date in zip(work_df["ticker"], work_df["ts"])
]

finite_prices = np.isfinite(work_df[all_price_cols]).all(axis=1)
positive_prices = (work_df[all_price_cols] > 0).all(axis=1)

volume_valid = (
    np.isfinite(work_df["volume"])
    & (work_df["volume"] >= 0)
)

key_valid = work_df["ticker"].notna() & work_df["ts"].notna()
duplicate_key = work_df.duplicated(["ticker", "ts"], keep=False)

quarantine_mask = (
    pd.Series(invalid_rows, index=work_df.index)
    | ~finite_prices
    | ~positive_prices
    | ~volume_valid
    | ~key_valid
    | duplicate_key
)

quarantine_df = work_df.loc[quarantine_mask].copy()
ohlc_hw_cleaned = work_df.loc[~quarantine_mask].copy()

print("Rows quarantined:", len(quarantine_df))
print("Rows retained:", len(ohlc_hw_cleaned))
print(
    "Extreme-return flags retained:",
    int(ohlc_hw_cleaned["extreme_return_flag"].sum())
)

Rows quarantined: 58926
Rows retained: 338785
Extreme-return flags retained: 5277


In [53]:

for col in all_price_cols + ["volume"]:
    ohlc_hw_cleaned[col] = pd.to_numeric(ohlc_hw_cleaned[col], errors="coerce")

price_values = ohlc_hw_cleaned[all_price_cols].to_numpy(dtype=float)
volume_values = ohlc_hw_cleaned["volume"].to_numpy(dtype=float)

valid_ohlc = (
    (ohlc_hw_cleaned["low"] <= ohlc_hw_cleaned["open"])
    & (ohlc_hw_cleaned["open"] <= ohlc_hw_cleaned["high"])
    & (ohlc_hw_cleaned["low"] <= ohlc_hw_cleaned["close"])
    & (ohlc_hw_cleaned["close"] <= ohlc_hw_cleaned["high"])
)

parsed_dates = pd.to_datetime(ohlc_hw_cleaned["ts"], errors="coerce")
valid_iso_dates = (
    parsed_dates.notna()
    & parsed_dates.dt.strftime("%Y-%m-%d").eq(ohlc_hw_cleaned["ts"].astype(str))
)

validation_results = {
    "Missing price values": int(ohlc_hw_cleaned[all_price_cols].isna().sum().sum()),
    "Non-finite price values": int((~np.isfinite(price_values)).sum()),
    "Nonpositive price values": int((price_values <= 0).sum()),
    "Duplicate (ticker, ts) rows": int(
        ohlc_hw_cleaned.duplicated(["ticker", "ts"]).sum()
    ),
    "Missing ticker values": int(ohlc_hw_cleaned["ticker"].isna().sum()),
    "Invalid dates": int((~valid_iso_dates).sum()),
    "Missing volume values": int(ohlc_hw_cleaned["volume"].isna().sum()),
    "Non-finite volume values": int((~np.isfinite(volume_values)).sum()),
    "Negative volume values": int((volume_values < 0).sum()),
    "Invalid OHLC relationships": int((~valid_ohlc).sum()),
}

print("VALIDATION RESULTS")
for check, count in validation_results.items():
    print(f"{check}: {count}")

assert all(count == 0 for count in validation_results.values()), (
    "Validation failed. Review the checks above before saving the cleaned table."
)

print("\nAll validation checks passed.")


VALIDATION RESULTS
Missing price values: 0
Non-finite price values: 0
Nonpositive price values: 0
Duplicate (ticker, ts) rows: 0
Missing ticker values: 0
Invalid dates: 0
Missing volume values: 0
Non-finite volume values: 0
Negative volume values: 0
Invalid OHLC relationships: 0

All validation checks passed.


In [54]:
def price_statistics(data, label):
    rows = []

    for col in all_price_cols + ["volume"]:
        values = pd.to_numeric(data[col], errors="coerce")
        finite_values = values[np.isfinite(values)]

        rows.append({
            "dataset": label,
            "column": col,
            "count": int(finite_values.count()),
            "missing_or_nonfinite": int(len(values) - finite_values.count()),
            "mean": finite_values.mean(),
            "std": finite_values.std(),
            "min": finite_values.min(),
            "max": finite_values.max(),
        })

    return rows


stats_before = price_statistics(raw_df, "Before cleaning")
stats_after = price_statistics(ohlc_hw_cleaned, "After cleaning")

stats_df = pd.DataFrame(stats_before + stats_after)
display(stats_df)


,dataset,column,count,missing_or_nonfinite,mean,std,min,max
0,Before cleaning,open,397711,0,4.628548e+01,6.713259e+01,0.000000,6.042600e+02
1,Before cleaning,high,397711,0,4.674766e+01,6.777821e+01,0.005208,6.086300e+02
2,Before cleaning,low,397711,0,4.581628e+01,6.647718e+01,0.004801,5.978800e+02
3,Before cleaning,close,397711,0,4.629290e+01,6.714188e+01,0.005208,6.054000e+02
4,Before cleaning,adj_close,377856,19855,5.165225e+01,1.945840e+02,0.002945,1.044251e+04
5,Before cleaning,adj_close,377856,19855,5.165225e+01,1.945840e+02,0.002945,1.044251e+04
6,Before cleaning,volume,397711,0,3.097773e+07,1.266237e+08,0.000000,9.276606e+09
7,After cleaning,open,338785,0,4.633692e+01,6.715555e+01,0.005208,6.042600e+02
8,After cleaning,high,338785,0,4.679965e+01,6.780213e+01,0.005208,6.086300e+02
9,After cleaning,low,338785,0,4.586814e+01,6.650363e+01,0.004801,5.978800e+02


In [56]:
ohlc_hw_cleaned["ts"] = (
    pd.to_datetime(ohlc_hw_cleaned["ts"], errors="coerce")
    .dt.strftime("%Y-%m-%d")
)

conn = sqlite3.connect(DB_PATH)

try:
    ohlc_hw_cleaned.to_sql(
        "ohlc_hw_cleaned", conn, if_exists="replace", index=False
    )
finally:
    conn.close()
